# Mapa interactivo

In [1]:
import pandas as pd
import json
from keplergl import KeplerGl

/home/guoqing/Documentos/startups-analysis-spain/.venv/lib/python3.12/site-packages/keplergl/keplergl.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


In [2]:
def add_geojson_info(row):
    postal_code = row["codigo_postal"]
    geojson_path = f"../data/silver/geojson/{postal_code}_polygon.geojson"
    try:
        with open(geojson_path, "r", encoding="utf-8") as f:
            geojson_data = json.load(f)
        row["geojson_polygon"] = json.dumps(geojson_data)
    except FileNotFoundError:
        row["geojson_polygon"] = None
    return row
empresas_df = pd.read_parquet('../data/silver/iberinform_startups_certificadas.parquet', engine='pyarrow')[['fecha_constitucion','nif', 'denominacion', 'descripcion_seccion_cnae','descripcion_grupo_cnae','objeto_social', 'codigo_postal', 'latitud', 'longitud']]
empresas_df = empresas_df.apply(add_geojson_info, axis=1)
empresas_df.fecha_constitucion = pd.to_datetime(empresas_df.fecha_constitucion, errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
empresas_df = empresas_df.dropna(subset=['fecha_constitucion']).copy()
universidades_df = pd.read_csv(
    "../data/silver/universidades_arc_gis.csv", sep=";",
    dtype={"codigo_postal": str, "latitud": float, "longitud": float},
)
aei_df = pd.read_csv(
    "../data/silver/agrupaciones_empresariales_innovadoras.csv", sep=";",
    dtype={"codigo_postal": str, "latitud": float, "longitud": float},
)
centros_tecnologicos_df = pd.read_csv(
    "../data/silver/centros_tecnologicos_y_de_apoyo_recidi.csv", sep=";",
    dtype={"codigo_postal": str, "latitud": float, "longitud": float},
)
parques_df = pd.read_csv(
    "../data/silver/parques_cientificos_tecnologicos_recidi.csv", sep=";",
    dtype={"codigo_postal": str, "latitud": float, "longitud": float},
)
icts_df = pd.read_csv(
    "../data/silver/icts.csv", sep=";",
    dtype={"latitud": float, "longitud": float},
)
somma_df = pd.read_csv(
    "../data/silver/centros_somma_recidi.csv", sep=";",
    dtype={"latitud": float, "longitud": float},
)

In [3]:
print(
    somma_df.latitud.isna().sum(),
    icts_df.latitud.isna().sum(),
    parques_df.latitud.isna().sum(),
    centros_tecnologicos_df.latitud.isna().sum(),
    aei_df.latitud.isna().sum(),
    universidades_df.latitud.isna().sum(),
    empresas_df.latitud.isna().sum(),
    empresas_df.fecha_constitucion.isna().sum(),
)

0 0 0 0 0 0 0 0


In [10]:
mapa = KeplerGl(height=1000, 
                data={
                    "Empresas": empresas_df.query("fecha_constitucion >= '2020-12-31' and fecha_constitucion <= '2024-12-31'"), 
                    "Universidades": universidades_df, 
                    "Agrupaciones Empresariales Innovadoras": aei_df, 
                    "Centros Tecnológicos y de Apoyo": centros_tecnologicos_df, 
                    "Parques Científicos y Tecnológicos": parques_df, 
                    "ICTS": icts_df, 
                    "Centros y Unidades de Excelencia": somma_df})
mapa.config = {
    "version": "v1",
    "config": {
        "visState": {
            "filters": [
                {
                    "id": "empresas_filter",
                    "dataId": "Empresas",
                    "name": ["fecha_constitucion"],
                    "type": "timeRange",
                    "enlarged": True,
                    "plotType": "histogram",
                    "animationWindow": "incremental",
                }
            ],
            "layers": [
                {
                    "id": "empresas_layer",
                    "type": "heatmap",
                    "config": {
                        "dataId": "Empresas",
                        "label": "Intensificación por Año de Constitución",
                        "color": [241, 104, 19],
                        "columns": {
                            "lat": "latitud",
                            "lng": "longitud"
                        },
                        "isVisible": True,
                        "visConfig": {
                            "radius": 5,         # Ajusta el radio de influencia de cada punto (en píxeles)
                            "intensity": 1.5,     # Multiplicador de intensidad global
                            "gradient": [         # Paleta de colores (de frío a cálido)
                                "0000ff", "00ffff", "00ff00", "ffff00", "ff0000"
                            ],
                            "opacity": 0.8
                        }
                    }
                },
                {
                    "id": "universidades_layer",
                    "type": "point",
                    "config": {
                        "dataId": "Universidades",
                        "label": "Universidades",
                        "color": [0, 255, 0],
                        "columns": {
                            "lat": "latitud",
                            "lng": "longitud"
                        },
                        "isVisible": False,
                        "visConfig": {
                            "radius": 15,
                            "fixedRadius": False,
                            "opacity": 0.8,
                            "outline": False,
                            "thickness": 2,
                            "strokeColor": [255, 255, 255],
                            "colorRange": {
                                "name": "Global Warming",
                                "type": "sequential",
                                "category": "Uber",
                                "colors": ["#5A1846", "#900C3F", "#C70039", "#E3611C", "#F1920E", "#FFC300"]
                            },
                            "radiusRange": [0, 50],
                            "filled": True
                        }
                    }
                },
                {
                    "id": "parques_layer",
                    "type": "point",
                    "config": {
                        "dataId": "Parques Científicos y Tecnológicos",
                        "label": "Parques Científicos y Tecnológicos",
                        "color": [0, 0, 255],
                        "columns": {
                            "lat": "latitud",
                            "lng": "longitud"
                        },
                        "isVisible": False,
                        "visConfig": {
                            "radius": 15,
                            "fixedRadius": False,
                            "opacity": 0.8,
                            "outline": False,
                            "thickness": 2,
                            "strokeColor": [255, 255, 255],
                            "colorRange": {
                                "name": "Global Warming",
                                "type": "sequential",
                                "category": "Uber",
                                "colors": ["#5A1846", "#900C3F", "#C70039", "#E3611C", "#F1920E", "#FFC300"]
                            },
                            "radiusRange": [0, 50],
                            "filled": True
                        }
                    }
                },
                {
                    "id": "AEI_layer",
                    "type": "point",
                    "config": {
                        "dataId": "Agrupaciones Empresariales Innovadoras",
                        "label": "Agrupaciones Empresariales Innovadoras",
                        "color": [215, 120, 0],
                        "columns": {
                            "lat": "latitud",
                            "lng": "longitud"
                        },
                        "isVisible": False,
                        "visConfig": {
                            "radius": 15,
                            "fixedRadius": False,
                            "opacity": 0.8,
                            "outline": False,
                            "thickness": 2,
                            "strokeColor": [255, 255, 255],
                            "colorRange": {
                                "name": "Global Warming",
                                "type": "sequential",
                                "category": "Uber",
                                "colors": ["#5A1846", "#900C3F", "#C70039", "#E3611C", "#F1920E", "#FFC300"]
                            },
                            "radiusRange": [0, 50],
                            "filled": True
                        }
                    }
                },
                {
                    "id": "icts_layer",
                    "type": "point",
                    "config": {
                        "dataId": "ICTS",
                        "label": "ICTS",
                        "color": [255, 0, 255],
                        "columns": {
                            "lat": "latitud",
                            "lng": "longitud"
                        },
                        "isVisible": False,
                        "visConfig": {
                            "radius": 15,
                            "fixedRadius": False,
                            "opacity": 0.8,
                            "outline": False,
                            "thickness": 2,
                            "strokeColor": [255, 255, 255],
                            "colorRange": {
                                "name": "Global Warming",
                                "type": "sequential",
                                "category": "Uber",
                                "colors": ["#5A1846", "#900C3F", "#C70039", "#E3611C", "#F1920E", "#FFC300"]
                            },
                            "radiusRange": [0, 50],
                            "filled": True
                        }
                    }
                },
                {
                    "id": "somma_layer",
                    "type": "point",
                    "config": {
                        "dataId": "Centros y Unidades de Excelencia",
                        "label": "Centros y Unidades de Excelencia",
                        "color": [255, 165, 0],
                        "columns": {
                            "lat": "latitud",
                            "lng": "longitud"
                        },
                        "isVisible": False,
                        "visConfig": {
                            "radius": 15,
                            "fixedRadius": False,
                            "opacity": 0.8,
                            "outline": False,
                            "thickness": 2,
                            "strokeColor": [255, 255, 255],
                            "colorRange": {
                                "name": "Global Warming",
                                "type": "sequential",
                                "category": "Uber",
                                "colors": ["#5A1846", "#900C3F", "#C70039", "#E3611C", "#F1920E", "#FFC300"]
                            },
                            "radiusRange": [0, 50],
                            "filled": True
                        }
                    }
                },
            ],
            "interactionConfig": {
                "tooltip": {
                    "fieldsToShow": {
                        "Empresas": [
                            {"name": "fecha_constitucion", "format": None}
                        ]
                    },
                    "enabled": True
                }
            }
        }
    }
}
mapa

User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


/home/guoqing/Documentos/startups-analysis-spain/.venv/lib/python3.12/site-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


KeplerGl(config={'version': 'v1', 'config': {'visState': {'filters': [{'id': 'empresas_filter', 'dataId': 'Emp…

In [5]:
mapa.save_to_html()

Map saved to keplergl_map.html!
